In [ ]:
# Cell 1: Install dependencies
!pip install -qU langgraph langchain-google-genai pydantic

In [ ]:
# Cell 2: Load API key securely and build the multi-agent system
import operator
from typing import Annotated, TypedDict, List
from pydantic import BaseModel, Field
from google.colab import userdata

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# Fetch API key securely from Colab Secrets
api_key = userdata.get('GOOGLE_API_KEY')

# Initialize free Gemini 2.5 Flash model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0
)

# 1. State Definition
class TaxSystemState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    next_agent: str

# Schema for structured supervisor decision
class RouterResponse(BaseModel):
    next_agent: str = Field(
        description="The next specialized agent to call: 'Extractor', 'Deductions', 'Calculator', or 'FINISH'"
    )

# 2. Agent Nodes
def supervisor_node(state: TaxSystemState):
    """Orchestrator node routing execution to specialist agents."""
    system_prompt = (
        "You are the Tax Orchestration Supervisor. Coordinate the tax agent workflow:\n"
        "1. 'Extractor': Extracts income, expenses, and filing status from user input.\n"
        "2. 'Deductions': Evaluates potential tax credits, deductions, and tax-saving strategies.\n"
        "3. 'Calculator': Calculates estimated tax liability and provides a final breakdown.\n\n"
        "Examine the execution history. Route to the next required agent. "
        "If all analysis and calculations are complete, output 'FINISH'."
    )
    
    structured_router = llm.with_structured_output(RouterResponse)
    decision = structured_router.invoke([SystemMessage(content=system_prompt)] + state["messages"])
    return {"next_agent": decision.next_agent}

def extractor_agent(state: TaxSystemState):
    system_prompt = "Extract gross income, business expenses, retirement contributions, and filing status."
    response = llm.invoke([SystemMessage(content=system_prompt)] + state["messages"])
    return {"messages": [HumanMessage(content=f"[Extractor Agent]: {response.content}", name="Extractor")]}

def deductions_agent(state: TaxSystemState):
    system_prompt = "Analyze eligible deductions (standard vs itemized) and applicable tax credits based on the extracted data."
    response = llm.invoke([SystemMessage(content=system_prompt)] + state["messages"])
    return {"messages": [HumanMessage(content=f"[Deductions Agent]: {response.content}", name="Deductions")]}

def calculator_agent(state: TaxSystemState):
    system_prompt = "Estimate final taxable income, marginal tax bracket, and total estimated tax liability based on extracted data and deductions."
    response = llm.invoke([SystemMessage(content=system_prompt)] + state["messages"])
    return {"messages": [HumanMessage(content=f"[Calculator Agent]: {response.content}", name="Calculator")]}

# 3. Assemble LangGraph Workflow
workflow = StateGraph(TaxSystemState)

workflow.add_node("Supervisor", supervisor_node)
workflow.add_node("Extractor", extractor_agent)
workflow.add_node("Deductions", deductions_agent)
workflow.add_node("Calculator", calculator_agent)

workflow.add_edge(START, "Supervisor")

workflow.add_conditional_edges(
    "Supervisor",
    lambda state: state["next_agent"],
    {
        "Extractor": "Extractor",
        "Deductions": "Deductions",
        "Calculator": "Calculator",
        "FINISH": END
    }
)

workflow.add_edge("Extractor", "Supervisor")
workflow.add_edge("Deductions", "Supervisor")
workflow.add_edge("Calculator", "Supervisor")

tax_graph = workflow.compile()
print("LangGraph Tax Multi-Agent Workflow compiled successfully!")

In [ ]:
# Cell 3: (Optional) Visualize the LangGraph workflow structure in Colab
from IPython.display import Image, display

try:
    display(Image(tax_graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
# Cell 4: Execute query through orchestrator
user_input = (
    "I filed Single in 2025. W2 gross income was $115,000. "
    "I contributed $6,500 to a Traditional IRA, paid $2,500 in student loan interest, "
    "and made $1,000 in charitable contributions. What is my tax breakdown?"
)

inputs = {"messages": [HumanMessage(content=user_input)]}

print("--- Running Multi-Agent Execution ---\n")
for step in tax_graph.stream(inputs):
    for node_name, state_update in step.items():
        print(f"📌 Step: {node_name}")
        if "messages" in state_update:
            print(state_update["messages"][-1].content)
        elif "next_agent" in state_update:
            print(f"Routing Decision -> Next Target: {state_update['next_agent']}")
        print("=" * 60)